# Tratamento, Higienização e Enriquecimento - Camada Silver
Padronização de tipos de dados, cálculo de retornos diários, volatilidade e médias móveis (SMA 7 e SMA 21).


In [ ]:
import os
import pandas as pd
import numpy as np

DATA_DIR = '/opt/airflow/data/stocks'
raw_path = os.path.join(DATA_DIR, 'raw_stocks_bronze.csv')
df = pd.read_csv(raw_path)
print(f"Carregados {len(df)} registros da camada Bronze.")


In [ ]:
# Conversão de tipos e ordenação cronológica
df['date'] = pd.to_datetime(df['date'])
df.sort_values(by=['ticker', 'date'], inplace=True)

# Cálculo de Métricas Técnicas
df['daily_return_pct'] = df.groupby('ticker')['close'].pct_change() * 100
df['sma_7'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
df['sma_21'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(window=21, min_periods=1).mean())
df['volatility_7d'] = df.groupby('ticker')['daily_return_pct'].transform(lambda x: x.rolling(window=7, min_periods=1).std())

# Limpeza de nulos resultantes de janelas móveis
df.fillna({'daily_return_pct': 0.0, 'volatility_7d': 0.0}, inplace=True)

silver_path = os.path.join(DATA_DIR, 'stocks_silver_refined.csv')
df.to_csv(silver_path, index=False)
print(f"Camada Silver persistida com sucesso em: {silver_path}")
df.head()
